# 2.5 Υπολογισμός μετρικών αξιολόγησης

## Φόρτωση δεδομένων

In [ ]:
# FLORES
with open("facebook_nllb_200_translations.txt", "r", encoding="utf-8") as f:
    fb_nllb = f.read().splitlines()

with open("Helsinki_NLP_Translations.txt", "r", encoding="utf-8") as f:
    helsinki_nlp = f.read().splitlines()

with open("Flores_ell.txt", "r", encoding="utf-8") as f:
    source = f.read().splitlines()

with open("Flores_eng.txt", "r", encoding="utf-8") as f:
    ref = f.read().splitlines()

# Test set
with open("facebook_nllb_200_test_translations.txt", "r", encoding="utf-8") as f:
    fb_nllb_test = f.read().splitlines()

with open("Helsinki_NLP_Test_Translations.txt", "r", encoding="utf-8") as f:
    helsinki_nlp_test = f.read().splitlines()

with open("data/test.el", "r", encoding="utf-8") as f:
    test_source = f.read().splitlines()

with open("data/test.en", "r", encoding="utf-8") as f:
    test_ref = f.read().splitlines()

## 2.5.1 BLEU και chrF

In [14]:
!pip install -q evaluate sacrebleu

In [15]:

import evaluate

In [ ]:
BLEU = evaluate.load("sacrebleu")
CHRF = evaluate.load("chrf")

### facebook/nllb-200-distilled-600M

In [ ]:
bleu_nllb_flores = BLEU.compute(predictions=fb_nllb, references=[[r] for r in ref])["score"]
chrf_nllb_flores = CHRF.compute(predictions=fb_nllb, references=[[r] for r in ref])["score"]

print("FLORES BLEU:", bleu_nllb_flores)
print("FLORES chrF:", chrf_nllb_flores)

BLEU: 33.96742699443383
chrF: 59.75878770624221


In [ ]:
bleu_nllb_test = BLEU.compute(predictions=fb_nllb_test, references=[[r] for r in test_ref])["score"]
chrf_nllb_test = CHRF.compute(predictions=fb_nllb_test, references=[[r] for r in test_ref])["score"]

print("Test set BLEU:", bleu_nllb_test)
print("Test set chrF:", chrf_nllb_test)

### Helsinki-NLP/opus-mt-grk-en

In [ ]:
bleu_helsinki_flores = BLEU.compute(predictions=helsinki_nlp, references=[[r] for r in ref])["score"]
chrf_helsinki_flores = CHRF.compute(predictions=helsinki_nlp, references=[[r] for r in ref])["score"]

print("FLORES BLEU:", bleu_helsinki_flores)
print("FLORES chrF:", chrf_helsinki_flores)

BLEU: 27.383005815157855
chrF: 56.11597367050298


In [ ]:
bleu_helsinki_test = BLEU.compute(predictions=helsinki_nlp_test, references=[[r] for r in test_ref])["score"]
chrf_helsinki_test = CHRF.compute(predictions=helsinki_nlp_test, references=[[r] for r in test_ref])["score"]

print("Test set BLEU:", bleu_helsinki_test)
print("Test set chrF:", chrf_helsinki_test)

## 2.5.2 COMET

In [ ]:
!pip install -q "transformers<5" unbabel-comet

In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

In [ ]:
from huggingface_hub import login
login()

In [ ]:
comet = evaluate.load("comet")

In [ ]:
# FLORES
comet_nllb_flores = comet.compute(predictions=fb_nllb, references=ref, sources=source)["mean_score"]
comet_helsinki_flores = comet.compute(predictions=helsinki_nlp, references=ref, sources=source)["mean_score"]

print("FLORES Facebook NLLB COMET:", comet_nllb_flores)
print("FLORES Helsinki NLP COMET:", comet_helsinki_flores)

In [ ]:
# Test set
comet_nllb_test = comet.compute(predictions=fb_nllb_test, references=test_ref, sources=test_source)["mean_score"]
comet_helsinki_test = comet.compute(predictions=helsinki_nlp_test, references=test_ref, sources=test_source)["mean_score"]

print("Test set Facebook NLLB COMET:", comet_nllb_test)
print("Test set Helsinki NLP COMET:", comet_helsinki_test)

In [ ]:
import gc
import torch

del comet
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

## 2.5.3 Συνοπτικός πίνακας αποτελεσμάτων

In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    {"Dataset": "FLORES", "Model": "facebook/nllb-200-distilled-600M", "BLEU": bleu_nllb_flores, "chrF": chrf_nllb_flores, "COMET": comet_nllb_flores},
    {"Dataset": "FLORES", "Model": "Helsinki-NLP/opus-mt-grk-en", "BLEU": bleu_helsinki_flores, "chrF": chrf_helsinki_flores, "COMET": comet_helsinki_flores},
    {"Dataset": "Test set", "Model": "facebook/nllb-200-distilled-600M", "BLEU": bleu_nllb_test, "chrF": chrf_nllb_test, "COMET": comet_nllb_test},
    {"Dataset": "Test set", "Model": "Helsinki-NLP/opus-mt-grk-en", "BLEU": bleu_helsinki_test, "chrF": chrf_helsinki_test, "COMET": comet_helsinki_test},
])

results_df[["BLEU", "chrF"]] = results_df[["BLEU", "chrF"]].round(2)
results_df["COMET"] = results_df["COMET"].round(4)
results_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
datasets = ["FLORES", "Test set"]
models = ["facebook/nllb-200-distilled-600M", "Helsinki-NLP/opus-mt-grk-en"]
short_names = ["NLLB-200", "Helsinki-NLP"]
x = range(len(datasets))
width = 0.35

for ax, metric in zip(axes, ["BLEU", "chrF", "COMET"]):
    for idx, (model, label) in enumerate(zip(models, short_names)):
        vals = [results_df[(results_df["Dataset"] == d) & (results_df["Model"] == model)][metric].values[0] for d in datasets]
        bars = ax.bar([p + idx * width for p in x], vals, width, label=label)
        fmt = "{:.4f}" if metric == "COMET" else "{:.2f}"
        ax.bar_label(bars, labels=[fmt.format(v) for v in vals], padding=3, fontsize=9)
    ax.set_title(metric)
    ax.set_xticks([p + width / 2 for p in x])
    ax.set_xticklabels(datasets)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].legend()
fig.suptitle("Evaluation Summary (EL -> EN)")
plt.tight_layout()
plt.savefig("evaluation_summary.png", dpi=150, bbox_inches="tight")
plt.show()